# Uzbekistan Economic Analysis (2010–2024)

This notebook presents a reproducible analysis of Uzbekistan’s macroeconomic performance from 2010 to 2024. It is structured for a senior-level audience and emphasizes data quality, transparent methodology, and clear economic interpretation.

Key objectives:
- Evaluate the national GDP trajectory and identify the main drivers of growth.
- Assess the role of fixed capital investment, external trade, employment, and productivity.
- Analyze regional economic patterns, inequality, and convergence/divergence dynamics.
- Surface realistic limitations and next steps for stronger evidence.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

This notebook loads the gold-layer fact_economic dataset and joins it to metric and region dimensions. The goal is to create a well-defined national and regional panel for 2010–2024 analysis.


In [ ]:
from pathlib import Path

print("Current working directory:", Path.cwd())

print("If this is not the repository root, set the notebook working directory to the project folder.")

In [ ]:
# Use centralized loader (scripts/load_data.py) to prepare the analysis panel
from scripts.load_data import load_panel, pivot_national
df, dim_metric, dim_region, fact_enriched = load_panel(data_root='data/gold', start=2010, end=2024, deflator_metric=None)
df.head()

We load the gold-layer records, including region and metric metadata, then build a clean analytics dataset.


In [ ]:
# Metadata already loaded via load_panel: dim_metric and dim_region are available.
# fact_enriched contains the joined fact + metadata if needed.
fact = fact_enriched  # ensure legacy cells referencing `fact` work

We merge the raw fact records with metric definitions and region metadata to produce a labeled dataset that is ready for analysis.


In [ ]:
fact_enriched = fact.merge(
    dim_metric,
    on="metric_id",
    how="left"
)

In [ ]:
fact_enriched = fact_enriched.merge(
    dim_region,
    on="region_code",
    how="left"
)

In [ ]:
fact_enriched

Keep only the columns needed for the analysis.

In [ ]:
df = fact_enriched[[
    "region_name",
    "year",
    "metric_code",
    "value",
    "unit"
]]

In [ ]:
df

## Executive summary (quick)

- **Top 3 KPIs:** Total nominal GDP growth (2010→2024), Average annual investment growth, Regional Gini (2024).
- **Top takeaways:** Productivity-led growth, investment-driven acceleration post-2016, rising regional divergence.

In [ ]:
# Executive summary KPIs and mini charts
from scripts.load_data import load_panel, pivot_national
from scripts.metrics import compute_productivity, gini_over_time
import matplotlib.pyplot as plt

df, dim_metric, dim_region, fact_enriched = load_panel('data/gold', 2010, 2024, deflator_metric=None)
# national pivot
nat = df[df['region_name']=='Republic of Uzbekistan'].pivot_table(index='year', columns='metric_code', values='real_value' if 'real_value' in df.columns else 'value')
gdp_2010 = nat.loc[2010, 'regional_gdp']
gdp_2024 = nat.loc[2024, 'regional_gdp']
gdp_growth_total = (gdp_2024 / gdp_2010 - 1) * 100
inv_growth_avg = nat['fixed_capital_investment'].pct_change().mean() * 100
# regional gini
regional_gdp = df[df['metric_code']=='regional_gdp'].pivot_table(index='year', columns='region_name', values='value')
from scripts.metrics import gini
gini_2024 = gini(regional_gdp.loc[2024].values)

print(f'Total nominal GDP growth (2010→2024): {gdp_growth_total:.1f}%')
print(f'Average annual investment growth: {inv_growth_avg:.1f}%')
print(f'Regional Gini (2024): {gini_2024:.3f}')

# mini charts: GDP time series and Gini time series
prod = compute_productivity(df)
gini_ts = gini_over_time(regional_gdp)

fig, axes = plt.subplots(1,2, figsize=(12,4))
axes[0].plot(nat.index, nat['regional_gdp'], marker='o')
axes[0].set_title('Uzbekistan GDP (Nominal)')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('GDP')

axes[1].plot(gini_ts.index, gini_ts['gini_coefficient'], marker='o', color='C1')
axes[1].set_title('Regional Gini (2010–2024)')
axes[1].set_xlabel('Year')

plt.tight_layout()
plt.show()